# Compute the AMOC time series

In [9]:
print('Loading packages...')
import sys
sys.path.append('../00_modules/.')
from import_packages import PackageGetter
globals().update(PackageGetter.import_standard_packages_for_analysis_and_plotting())
globals().update(PackageGetter.import_custom_packages())

Loading packages...


## 0) define functions

In [55]:
def calc_overturning_strength(
    da,
    freq_input,
    freq_output,
    basin_choice='atlantic_arctic_ocean',
    lat_choice=26.5,
    min_depth_choice=300,
    moc_component="Eulerian Mean",
    include_bolus=False,
    include_submeso=False,
):
    """
    AMOC / overturning strength from CMIP6 or CESM2 MOC output.
    """

    # ============================================================
    # units
    # ============================================================

    unit = da.attrs.get("units", "")

    # ============================================================
    # CESM2: fix moc_z units if needed
    # ============================================================

    if "moc_z" in da.coords:
        try:
            if float(da.moc_z.max()) > 20000:
                da = da.assign_coords(moc_z=da.moc_z / 100.0)
                da.moc_z.attrs["units"] = "m"
        except:
            pass

    # ============================================================
    # MOC component handling (CESM2)
    # ============================================================

    moc_dim = None
    if "moc_comp" in da.dims:
        moc_dim = "moc_comp"
    elif "moc_components" in da.dims:
        moc_dim = "moc_components"

    if moc_dim is not None:

        raw = da[moc_dim].values

        # decode components
        if np.issubdtype(raw.dtype, np.number):

            comp_map = {
                0: "Eulerian Mean",
                1: "Eddy-Induced (bolus)",
                2: "Submeso",
            }

            comps = np.array([comp_map.get(int(v), str(v)) for v in raw])

        else:
            try:
                comps = np.array([
                    x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
                    for x in raw
                ])
            except:
                comps = raw.astype(str)

        comps = np.char.strip(comps.astype(str))

        da = da.assign_coords({moc_dim: comps})

        # select components
        to_use = [moc_component]
        if include_bolus:
            to_use.append("Eddy-Induced (bolus)")
        if include_submeso:
            to_use.append("Submeso")

        sel = []
        for c in to_use:
            if c in da[moc_dim].values:
                sel.append(da.sel({moc_dim: c}))

        if len(sel) == 0:
            raise ValueError(f"No MOC components found. Available: {da[moc_dim].values}")

        da = sum(sel)

    # ============================================================
    # CESM2 basin handling (INTEGER-BASED FIXED)
    # ============================================================
    
    if "transport_reg" in da.dims:
    
        regs = da.transport_reg.values
    
        # ensure integers
        try:
            regs = regs.astype(int)
        except:
            regs = np.array([int(r) for r in regs])
    
        print("Available transport_reg indices:", regs)
    
        # CESM2 convention (your case)
        basin_map = {
            "global_ocean": 0,
            "atlantic_arctic_ocean": 1,
            "atlantic": 1,
        }
    
        if basin_choice not in basin_map:
            raise ValueError(f"Unknown basin_choice={basin_choice}")
    
        target = basin_map[basin_choice]
    
        if target not in regs:
            raise ValueError(
                f"Requested basin index {target} not in available {regs}"
            )
    
        print(f"... selecting transport_reg = {target}")
    
        da = da.sel(transport_reg=target)
    
        print("Remaining dims:", da.dims)

    # ============================================================
    # CMIP-style basin fallback
    # ============================================================

    elif "basin" in da.coords:

        basin_vals = da["basin"].values

        try:
            basin_vals = np.array([
                x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
                for x in basin_vals
            ])
        except:
            basin_vals = basin_vals.astype(str)

        da = da.assign_coords(basin=basin_vals)
        da = da.sel(basin=basin_choice)

    # ============================================================
    # mask invalid values
    # ============================================================

    da = xr.where(np.abs(da) < 1e35, da, np.nan)

    # ============================================================
    # latitude selection
    # ============================================================

    if "lat" in da.coords:
        da = da.sel(lat=lat_choice, method="nearest")
    elif "latitude" in da.coords:
        da = da.sel(latitude=lat_choice, method="nearest")
    elif "lato2" in da.coords:
        da = da.sel(lato2=lat_choice, method="nearest")
    elif "rlat" in da.coords:
        da = da.sel(rlat=lat_choice, method="nearest")
    elif "lat_aux_grid" in da.coords:
        da = da.sel(lat_aux_grid=lat_choice, method="nearest")
    else:
        raise ValueError("No known latitude coordinate found")

    # ============================================================
    # time averaging
    # ============================================================

    if freq_input != freq_output:
        da = TimeOperator.calc_temporal_mean_weighted(
            da,
            dim="time",
            freq_output=freq_output,
            weights=None,
            skip_if_native=True,
            verbosity=1
        ).compute()

    # ============================================================
    # overturning calculation
    # ============================================================

    if "lev" in da.dims:
        da = da.sel(lev=slice(min_depth_choice, None)).max("lev")

    elif "olevel" in da.dims:
        da = da.sel(olevel=slice(min_depth_choice, None)).max("olevel")

    elif "zoce" in da.dims:
        da = da.sel(zoce=slice(min_depth_choice, None)).max("zoce")

    elif "moc_z" in da.dims:
        da = da.sel(moc_z=slice(min_depth_choice, None)).max("moc_z")

    else:
        raise ValueError(f"No vertical coordinate found. Dims: {da.dims}")

    # ============================================================
    # metadata
    # ============================================================

    lat_string = str(lat_choice).replace(".", "p")

    da.name = f"amoc_at_{lat_string}"
    da.attrs["units"] = unit

    return da

## 1) get the varias, models and runs over which to do computation

In [60]:
lat_choice = 26.5
min_depth_choice = 300
basin_choice = 'atlantic_arctic_ocean'

freq_input = 'monthly'#, 'yearly', 'daily'] #freq_output = 'monthly'#, 'yearly', 'daily', 'climatology', None]
freq_output = 'yearly'
#varias = ['msftyz']#['msftyz']#,'msftmz']
models = ['MIROC-ES2L']#['CESM2']#['MIROC-ES2L']#['ACCESS-ESM1-5']#['EC-Earth3-ESM-1']#['UKESM1-2']#['GISSE2.1-G-CC2'] #['IPSL-CM6-ESMCO2']#['GFDL-ESM2M']#['NorESM2-LM']##['IPSL-CM6-ESMCO2']#,'NorESM2-LM','GFDL-ESM2M'] # ,['GFDL-ESM2M']#
runs = pruns.get_run_list('tipmip_tier1')#[:1]#[1:]#[:-1]#[1:2]#[:1]#[1:]#[1:2]#[:1]#[1:]#[-1:] #

def get_varia(model):
    if model in ['IPSL-CM6-ESMCO2','GFDL-ESM2M','GISSE2.1-G-CC2','UKESM1-2','EC-Earth3-ESM-1','ACCESS-ESM1-5']:
        varia = 'msftyz'
    elif model in ['NorESM2-LM']:
        varia = 'msftmz'
    elif model in ['CESM2']:
        varia = 'MOC'
    else:
        raise Exception('Varia not yet chosen for this model.')
    return varia

## 2) loop over models and runs to do the computation, plotting and saving

In [61]:
for model in models:

    varia = get_varia(model)

    mgrab = MODELgrabber.get_grabber(model)

    member = mgrab.get_member()

    global_stats = dict()

    for run in runs:

        print(f'Processing data for {varia}, {model}, {run}...')

        # ==========================================================
        # load data
        # ==========================================================

        da = mgrab.get_data(
            varia,
            run,
            freq_input=freq_input,
            verbose_level=1,
        )

        print(f'... loading {da.time.size} data points in time.')

        # ==========================================================
        # CESM2-specific preprocessing
        # ==========================================================

        if model == "CESM2":

            # ----------------------------------------------
            # convert moc_z from cm -> m if needed
            # ----------------------------------------------

            if "moc_z" in da.coords:

                try:

                    zmax = float(da.moc_z.max())

                    if zmax > 2e4:

                        print('... converting moc_z from cm to m')

                        da = da.assign_coords(
                            moc_z=da.moc_z / 100.
                        )

                        da.moc_z.attrs["units"] = "m"

                except Exception as e:

                    print(f'Warning: moc_z conversion failed: {e}')

        # ==========================================================
        # calculate overturning
        # ==========================================================

        if model == "CESM2":

            overturning = calc_overturning_strength(
                da,
                freq_input,
                freq_output,
                basin_choice=basin_choice,
                lat_choice=lat_choice,
                min_depth_choice=min_depth_choice,

                # CESM2 residual overturning
                moc_component="Eulerian Mean",
                include_bolus=True,
                include_submeso=False,
            )

        else:

            overturning = calc_overturning_strength(
                da,
                freq_input,
                freq_output,
                basin_choice=basin_choice,
                lat_choice=lat_choice,
                min_depth_choice=min_depth_choice,
            )

        lat_string = str(lat_choice).replace('.', 'p')

        # ==========================================================
        # save
        # ==========================================================

        save_dir = (
            f'./../01_postprocessed_data/'
            f'amoc_time_series/'
            f'{varia}/{model}/{run}/{member}/{freq_output}'
        )

        os.makedirs(save_dir, exist_ok=True)

        save_string = (
            f'{save_dir}/'
            f'{varia}_{model}_{run}_{member}_'
            f'{freq_output}_{basin_choice}_'
            f'{lat_string}_{min_depth_choice}m.nc'
        )

        print(f'... saving under {save_string}')

        overturning.to_netcdf(save_string)

        # ==========================================================
        # keep for plotting
        # ==========================================================

        global_stats[run] = overturning

        # ==========================================================
        # plot
        # ==========================================================

        print(f'... plotting {run}')

        global_stats[run].plot(
            label=run,
            linewidth=2,
        )

        print(' ')

    plt.legend()

    plt.title(
        f'{model} AMOC '
        f'({lat_choice}N, >{min_depth_choice}m)'
    )

    plt.ylabel(overturning.attrs.get('units', ''))

    plt.grid(alpha=0.3)

    plt.show()

Exception: Varia not yet chosen for this model.

## Old way of doing it (before uncmorized CESM2 was included)

In [ ]:
def calc_overturning_strength(da, freq_input, freq_output, basin_choice='atlantic_arctic_ocean', lat_choice=26.5, min_depth_choice=300):

    # get the unit
    unit = da.units
    
    # treat the basin names for better consistency across models
    if 'sector' in da.coords:
        sector_clean = np.char.decode(da.sector.values, "utf-8")
        sector_clean = np.char.strip(sector_clean)
        print(sector_clean)
        da = da.assign_coords(basin=("basin", sector_clean))    
    elif 'basin' in da.coords:
        print(da.basin.values)
        try:
            sector_clean = np.char.decode(da.basin.values, "utf-8")
            sector_clean = np.char.strip(sector_clean)
        except:
            sector_clean = da.basin.values
        da = da.assign_coords(basin=("basin", sector_clean))
        

    # make basin an index to choose from and then choose from it
    if "basin" in da.coords:
        bbs = da.basin
        da = da.set_index(basin="basin")

    if "basin" in da.coords:
        da = da.sel(basin=basin_choice)
    else:
        da = da
        
    # mask out values
    da2 = xr.where(da<1e35,da,np.nan)

    # get the latitude dimension and choose the latitude
    if "lat" in da.coords:
        latdim = "lat"
        da3 = da2.sel(lat=lat_choice,method='nearest')
    elif "latitude" in da.coords:
        latdim = "latitude"
        da3 = da2.sel(latitude=lat_choice,method='nearest')
    elif "lato2" in da.coords:
        latdim = 'lato2'
        da3 = da2.sel(lato2=lat_choice,method='nearest')
    elif "rlat" in da.coords:
        latdim = 'rlat'
        da3 = da2.sel(rlat=lat_choice,method='nearest')
    else:
        print(da.dims)
        raise Exception('No known latitude dim.')
        
    # adjust time to freq_output
    if freq_input != freq_output:
        print('Need to adjust the time axis...')
        da4 = TimeOperator.calc_temporal_mean_weighted(da3,dim="time",freq_output=freq_output,          # None, "monthly", "yearly", "climatology"
        weights=None,
        skip_if_native=True,
        verbosity=1).compute()
    else:
        da4 = da3

    # now compute the overturning time series
    print('get time series')

    if "lev" in da.dims:
        levdim = "lev"
        overturning = da4.sel(lev=slice(min_depth_choice,10000)).max(dim='lev')
    elif "olevel" in da.dims:
        levdim = "olevel"
        overturning = da4.sel(olevel=slice(min_depth_choice,10000)).max(dim='olevel')
    elif "zoce" in da.dims:
        levdim = "zoce"
        overturning = da4.sel(zoce=slice(min_depth_choice,10000)).max(dim='zoce')        
    else:
        print(da.dims)
        raise Exception('Overturning could not be calculated')

    print(lat_choice)
    lat_string = str(lat_choice).replace('.','p')
    overturning.name = f"amoc_at_{lat_string}"
    overturning.attrs['units'] = unit

    return overturning


In [8]:
for model in models:
    varia = get_varia(model)
    mgrab = MODELgrabber.get_grabber(model)
    member = mgrab.get_member()
    
    global_stats = dict()
    for run in runs:
        print(f'Processing data for {varia}, {model}, {run}...')
        
        # first get the dataset
        #print('... loading ...')
        da = mgrab.get_data(varia,run,freq_input=freq_input,verbose_level=1)#.compute()
        print(f'... loading {da.time.size} data points in time.')

        # now calculate the overturning
        overturning = calc_overturning_strength(da, freq_input, freq_output, basin_choice=basin_choice, lat_choice=lat_choice, min_depth_choice=min_depth_choice)
        lat_string = str(lat_choice).replace('.','p')

        # now save the data
        save_dir = f'./../01_postprocessed_data/amoc_time_series/{varia}/{model}/{run}/{member}/{freq_output}'
        os.makedirs(save_dir,exist_ok=True)
        save_string = f'{save_dir}/{varia}_{model}_{run}_{member}_{freq_output}_{basin_choice}_{lat_string}_{min_depth_choice}m.nc'
        print(f'... saving under {save_string}')
        overturning.to_netcdf(save_string)#,encoding=encoding)
        
        # keep in a local dictionary for plotting
        global_stats[run] = overturning

        #for key in global_stats.keys():
        print(f'... plotting {run}')
        global_stats[run].plot()
        print(' ')
    plt.show()


Exception: Varia not yet chosen for this model.